In [ ]:
!git clone https://github.com/CryAndRRich/codapath.git

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

In [ ]:
# pathmnist | histoset | skintissue
DATASET = "histoset"

# random | coreset | typiclust | activeft | badge | entropy | margin
# codapath | scalpel | uncertainty_herding | tcm | dropquery | refine
SAMPLER_NAME = "codapath"

SEED = 42

In [ ]:
import os
from huggingface_hub import snapshot_download, login

login("YOUR_HUGGINGFACE_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Downloading facebook/dinov2-base...")
snapshot_download(repo_id="facebook/dinov2-base")

if SAMPLER_NAME == "codapath":
    print("Downloading vinid/plip (CODAPath image + text encoder)...")
    snapshot_download(repo_id="vinid/plip")

    print("Downloading BiomedNLP-PubMedBERT (CODAPath text encoder)...")
    snapshot_download(repo_id="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")

    print("Downloading BiomedCLIP (CODAPath image encoder)...")
    snapshot_download(repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")

# NOTE: SCALPEL (v6) needs only DINOv2 above. Its stain view is computed from raw
# H&E pixels via colour deconvolution — no extra model / download required.

In [ ]:
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
import yaml
import torch

from run import main

In [ ]:
PATHMNIST_PATH  = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH   = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

DATA_DICT = {
    "pathmnist":  PATHMNIST_PATH,
    "histoset":   HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

In [ ]:
CONFIG_PATH = "config/config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

training_cfg = config.get("training", {})
dataset_info = config["datasets"][DATASET]
sampler_cfg  = config.get("samplers", {}).get(SAMPLER_NAME, {})

In [ ]:
main(
    data_path=DATA_DICT[DATASET],
    sampler_name=SAMPLER_NAME,
    num_classes=dataset_info["num_classes"],
    cumulative_budget=config["cumulative_budget"],
    data_descriptions=dataset_info["descriptions"],
    prompt_templates=config["prompt_templates"],
    sampler_cfg=sampler_cfg,
    probe_epochs=training_cfg["probe_epochs"],
    probe_lr=training_cfg["probe_lr"],
    knn_k=training_cfg["knn_k"],
    knn_threshold=training_cfg["knn_threshold"],
    device=torch.device(config["device"]),
    random_seed=SEED,
    save_dir=f"checkpoints/{DATASET}",
    verbose=True,
    model_cfg=config.get("models", {}),
)